# pydantic的高级特性

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)
TONGYI_API_KEY = os.getenv("TONGYI_API_KEY")
TONGYI_BASE_URL = os.getenv("TONGYI_BASE_URL")

model=init_chat_model(
    model="deepseek-v4-flash",  # 模型名称
    model_provider="openai",
    api_key=TONGYI_API_KEY,
    base_url="https://llm-v2xyqhn297xtv6xq.cn-beijing.maas.aliyuncs.com/compatible-mode/v1"  # TONGYI API 的基础 URL
)

## 可选字段-Optional

In [6]:
from pydantic import BaseModel,Field
from typing import Optional


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age : Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

print(result)
print(type(result))

name='张三' age=28 occupation='软件工程师'
<class '__main__.Person'>


## 默认值

In [7]:
from pydantic import BaseModel,Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age : int = Field(default=10,description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

print(result)
print(type(result))

name='张三' age=28 occupation='软件工程师'
<class '__main__.Person'>


## 枚举类型

In [8]:
from enum import Enum


# 定义枚举类型
class Priority(str,Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"



class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")

print(result)


name='王小明' phone='138-1234-5678' email='' issue='订单未发货，客户很着急' urgency=<Priority.HIGH: '高'>


In [9]:
from typing import Literal


class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["低","中","高"] = Field(description="紧急程度")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")

print(result)

name='王小明' phone='138-1234-5678' email='' issue='订单未发货' urgency='高'


# 列表提取

In [10]:

from typing import List


class Person(BaseModel):
    """人物信息"""
    name : str = Field(description="姓名")
    age :int = Field(description="年龄")

class PersonList(BaseModel):
    """人物列表"""
    people : List[Person]  # 多个Person的对象


structured_model = model.with_structured_output(PersonList)

result = structured_model.invoke("张三 30岁, 李四 40岁")
print(result)

people=[Person(name='张三', age=30), Person(name='李四', age=40)]


# 嵌套结构

In [11]:
class Address(BaseModel):
    """地点描述"""
    city : str = Field(description="城市")
    district : str = Field(description="区域")


class Company(BaseModel):
    """公司信息"""
    name : str = Field(description="公司名称")
    address : Address = Field(description="公司所在地")

structured_model = model.with_structured_output(Company)

result = structured_model.invoke("阿里巴巴在杭州的滨江区")
print(result)

name='阿里巴巴（中国）有限公司滨江园区' address=Address(city='杭州', district='滨江区')


# 限制条件

In [12]:
from pydantic import ValidationError

class User(BaseModel):
    name : str = Field(description="姓名",min_length=2,max_length=50)
    age : int = Field(description="年龄",le=150)
    email : str = Field(description="邮箱")


try:
    user1 = User(name="tom",age = 20,email="tom@126.com")
    print(f"[OK]{user1}")
except ValidationError as e:
    print(f"[FAIL]{e}")

[OK]name='tom' age=20 email='tom@126.com'


In [15]:
from pydantic import ValidationError

class User(BaseModel):
    name : str = Field(description="姓名",min_length=2,max_length=50)
    age : int = Field(description="年龄",le=150)
    email : str = Field(description="邮箱")


try:
    user2 = User(age = 25,email="jerry@126.com")
    print(f"[OK]{user1}")
except ValidationError as e:
    print(f"[FAIL]{e}")

[FAIL]1 validation error for User
name
  Field required [type=missing, input_value={'age': 25, 'email': 'jerry@126.com'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
